In [1]:
import os
import sys
from pathlib import Path

# Locate the repo root as the directory containing this notebook.
# __vsc_ipynb_file__ is injected by VS Code; fall back to cwd for other environments.
try:
    NOTEBOOK_DIR = str(Path(__vsc_ipynb_file__).parent.resolve())
except NameError:
    NOTEBOOK_DIR = str(Path.cwd())

os.chdir(NOTEBOOK_DIR)
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

print(f"Working directory: {os.getcwd()}")

Working directory: E:\OneDrive - Umich\Work\Propagation\SWMF-IMF


In [2]:
import pandas as pd
from pyspedas import CDAWeb
from l1_pipeline import get_one_day_swmf_input, create_position_file
from l1_combine import create_combined_l1_files

cda = CDAWeb()

12-Mar-26 14:14:20: c:\Users\Connor\miniconda3\Lib\site-packages\spacepy\time.py:2448: UserWarning: Leapseconds may be out of date. Use spacepy.toolbox.update(leapsecs=True)
  _read_leaps()



In [3]:

# Force-reload pipeline modules so any code changes made this session
# are picked up without restarting the kernel.
import importlib
import l1_pipeline, l1_combine, l1_filters, l1_quality
for mod in [l1_filters, l1_quality, l1_pipeline, l1_combine]:
    importlib.reload(mod)
from l1_pipeline import get_one_day_swmf_input, create_position_file
from l1_combine import create_combined_l1_files
print('Modules reloaded.')


Modules reloaded.


In [ ]:
# -----------------------------------------------------------------------
# Date range to process.
# Uncomment the rolling-window block to always run the last N days.
# -----------------------------------------------------------------------

# today = pd.Timestamp.utcnow().strftime('%Y-%m-%d')
# days = pd.date_range(end=today, periods=7).strftime('%Y-%m-%d').tolist()

from plot_l1_may2024 import plot_day

days = pd.date_range(
    start='2024-05-01', end='2024-05-31'
).strftime('%Y-%m-%d').tolist()

start_time = pd.Timestamp.now()

# Seed the window: download the day before day-1 (context only) and day-1.
day_before = (pd.Timestamp(days[0]) -
              pd.Timedelta(days=1)).strftime('%Y-%m-%d')
get_one_day_swmf_input(day_before, cda)
get_one_day_swmf_input(days[0], cda)
create_position_file(days[0], cda)

# Crawling window: download tomorrow, combine today (yesterday + tomorrow
# as context), plot, then advance.  Each day is downloaded exactly once.
for i, day in enumerate(days):
    prev_day = day_before if i == 0 else days[i - 1]

    if i < len(days) - 1:
        next_day = days[i + 1]
        get_one_day_swmf_input(next_day, cda)
        create_position_file(next_day, cda)
    else:
        next_day = None

    create_combined_l1_files(day, prev_day=prev_day, next_day=next_day)
    try:
        plot_day(day)
    except Exception as exc:
        print(f"  Plot error for {day}: {exc}")
    print(f"Completed: {day}")

end_time = pd.Timestamp.now()
print(f"\nDone. Elapsed: {end_time - start_time}")

Querying CDAWeb (attempt 1/3)...


12-Mar-26 14:14:27: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240430_v07.cdf to E:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240430_v07.cdf


12-Mar-26 14:14:27: Download of E:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240430_v07.cdf complete, 0.235 MB in 0.4 sec (0.570 MB/sec) (transfer_normal)
12-Mar-26 14:14:27: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240501_v07.cdf to E:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240501_v07.cdf
12-Mar-26 14:14:27: Download of E:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240501_v07.cdf complete, 0.238 MB in 0.4 sec (0.673 MB/sec) (transfer_normal)
12-Mar-26 14:14:28: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/swepam/level_2_cdaweb/swe_h0/2024/ac_h0_swe_20240430_v11.cdf to E:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\swepam\level_2_cdaweb\swe_h0\2024\ac_h0_swe_20240430_v11.cdf
12-Mar-26 14:14:28: Download of E:\OneDrive - Umich\Work\P


Processing ace...
  -> Running Despike (Growth & Change limits)...
Saved L1/2024/04/30\L1_ace.dat
Cleaning up ace CDFs...

Processing DSCOVR (NGDC)...
  Downloaded oe_f1m_dscovr_s20240430000000_e20240430235959_p20240501021914_pub.nc.gz -> cdf_temp\dscovr_f1m_20240430.nc.gz
  Downloaded oe_m1m_dscovr_s20240430000000_e20240430235959_p20240501021331_pub.nc.gz -> cdf_temp\dscovr_m1m_20240430.nc.gz
  -> Running Despike (Growth & Change limits)...
Saved L1/2024/04/30\L1_dscovr.dat
  Cleaning up DSCOVR NGDC files...

Processing wind...
Rotating WIND Plasma data from GSE to GSM...
  -> Running Despike (Growth & Change limits)...
Saved L1/2024/04/30\L1_wind.dat
Cleaning up wind CDFs...
Querying CDAWeb (attempt 1/3)...


12-Mar-26 14:14:44: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240501_v07.cdf to E:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240501_v07.cdf


...

Saved L1/2024/05/06\L1_satpos.dat

Processing L1 data for 2024-05-05  (context window: ['2024-05-04', '2024-05-05', '2024-05-06'])...
  Running plasma quality assessment for all satellites...
  ACE quality: flagged bad: {'Ux': 0, 'Uy': 0, 'Uz': 221, 'rho': 0, 'T': 166}
  DSCOVR quality: flagged bad: {'Ux': 26, 'Uy': 1332, 'Uz': 1208, 'rho': 345, 'T': 2585}
  WIND quality: flagged bad: {'Ux': 0, 'Uy': 0, 'Uz': 27, 'rho': 0, 'T': 1026}
  -> Running Despike (Growth & Change limits)...
Created L1/2024/05/05\L1_combined.dat
  -> Propagating to 14 Re (89194 km)...
    Created L1/2024/05/05\IMF_14Re.dat
  -> Propagating to 32 Re (203872 km)...
    Created L1/2024/05/05\IMF_32Re.dat
Saved plots\2024_05_05.png
Completed: 2024-05-05
Querying CDAWeb (attempt 1/3)...


12-Mar-26 14:18:46: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240507_v07.cdf to E:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240507_v07.cdf


12-Mar-26 14:18:47: Download of E:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240507_v07.cdf complete, 0.232 MB in 0.3 sec (0.664 MB/sec) (transfer_normal)
12-Mar-26 14:18:47: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240508_v07.cdf to E:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240508_v07.cdf
12-Mar-26 14:18:47: Download of E:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240508_v07.cdf complete, 0.229 MB in 0.4 sec (0.523 MB/sec) (transfer_normal)
12-Mar-26 14:18:47: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/swepam/level_2_cdaweb/swe_h0/2024/ac_h0_swe_20240507_v11.cdf to E:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\swepam\level_2_cdaweb\swe_h0\2024\ac_h0_swe_20240507_v11.cdf
12-Mar-26 14:18:47: Download of E:\OneDrive - Umich\Work\P


Processing ace...
  -> Running Despike (Growth & Change limits)...
Saved L1/2024/05/07\L1_ace.dat
Cleaning up ace CDFs...

Processing DSCOVR (NGDC)...
  Downloaded oe_f1m_dscovr_s20240507000000_e20240507235959_p20240508022112_pub.nc.gz -> cdf_temp\dscovr_f1m_20240507.nc.gz
  Downloaded oe_m1m_dscovr_s20240507000000_e20240507235959_p20240508021533_pub.nc.gz -> cdf_temp\dscovr_m1m_20240507.nc.gz
  -> Running Despike (Growth & Change limits)...
Saved L1/2024/05/07\L1_dscovr.dat
  Cleaning up DSCOVR NGDC files...

Processing wind...
Rotating WIND Plasma data from GSE to GSM...
  -> Running Despike (Growth & Change limits)...
Saved L1/2024/05/07\L1_wind.dat
Cleaning up wind CDFs...

--- Generating L1_satpos.dat for 2024-05-07 ---
Querying Position Data (11:00-13:00 UT) (attempt 1/3)...


12-Mar-26 14:19:01: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240507_v07.cdf to E:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240507_v07.cdf
12-Mar-26 14:19:02: Download of E:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\ace\mag\level_2_cdaweb\mfi_h0\2024\ac_h0_mfi_20240507_v07.cdf complete, 0.232 MB in 0.4 sec (0.526 MB/sec) (transfer_normal)
12-Mar-26 14:19:02: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/wind/mfi/mfi_h0/2024/wi_h0_mfi_20240507_v05.cdf to E:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\wind\mfi\mfi_h0\2024\wi_h0_mfi_20240507_v05.cdf
12-Mar-26 14:19:02: Download of E:\OneDrive - Umich\Work\Propagation\SWMF-IMF\cdf_temp\wind\mfi\mfi_h0\2024\wi_h0_mfi_20240507_v05.cdf complete, 2.497 MB in 0.6 sec (3.917 MB/sec) (transfer_normal)
12-Mar-26 14:19:02: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/dscovr/orbit/pre_or/2024/dscovr_

In [ ]:
# -----------------------------------------------------------------------
# Context plots: each day with ±6 h of the neighbouring days.
# Vertical dashed lines mark midnight boundaries so you can immediately
# spot any jumps between days.  Shaded regions are the context windows.
# Output goes to  plots/with_context/
# Standalone — can be run without running cell 3 first.
# -----------------------------------------------------------------------

import pandas as pd
from plot_l1_may2024 import plot_day_with_context

days = pd.date_range(
    start='2024-05-01', end='2024-05-31'
).strftime('%Y-%m-%d').tolist()

context_start = pd.Timestamp.now()

for day in days:
    try:
        plot_day_with_context(day, context_hours=6,
                              output_dir='plots/with_context')
    except Exception as exc:
        print(f"  Context-plot error for {day}: {exc}")

context_end = pd.Timestamp.now()
print(f"\nContext plots done. Elapsed: {context_end - context_start}")